## S00 — Configuration and reproducibility setup

# Initializes:
- canonical project configuration
- input/output paths
- logging
- deterministic random seed
- numeric parsing and QC utilities
- shared helper functions used across S01–S11
- config snapshot for reproducibility

## Imports

In [ ]:
from __future__ import annotations

import json
import logging
import platform
import random
import re
from copy import deepcopy
from datetime import datetime
from pathlib import Path
from typing import Optional, Sequence, Dict, Any, Tuple

import numpy as np
import pandas as pd

## Base paths and config location

In [ ]:
BASE_WORKDIR = Path("/content")
CONFIG_PATH = BASE_WORKDIR / "config.json"

INPUT_DIR_CANDIDATES = [
    Path("/content"),
    Path("/mnt/data"),
]

## Input file resolver


In [ ]:
def resolve_uploaded_xlsx(
    preferred_names: Sequence[str] = ("patients_results.xlsx",)
) -> str:

    existing_dirs = [p for p in INPUT_DIR_CANDIDATES if p.exists()]

    if not existing_dirs:
        raise FileNotFoundError(
            "Cannot find any expected input directory. Checked: "
            + ", ".join(str(p) for p in INPUT_DIR_CANDIDATES)
        )


    for base_dir in existing_dirs:
        for name in preferred_names:
            p = base_dir / name
            if p.exists():
                return str(p)


    xlsx_files = []
    for base_dir in existing_dirs:
        xlsx_files.extend(list(base_dir.glob("*.xlsx")))

    xlsx_files = sorted(
        xlsx_files,
        key=lambda p: p.stat().st_mtime,
        reverse=True
    )

    if not xlsx_files:
        raise FileNotFoundError(
            "No .xlsx files found in expected locations: "
            + ", ".join(str(p) for p in existing_dirs)
        )

    if len(xlsx_files) > 1:
        logging.warning(
            "Multiple .xlsx files found; using most recent: %s",
            xlsx_files[0]
        )

    return str(xlsx_files[0])

## Default config

In [ ]:
CFG_DEFAULT: Dict[str, Any] = {
    "project": {
        "name": "PCOS_TAI_cross_sectional",
        "version": "0.2.1",
        "random_seed": 20260222,
        "analysis_report_standard": "Journal-ready reproducibility notes",
        "python_recommended": ">=3.10",
        "notebook": "S00_config"
    },

    "paths": {
        "input_xlsx": None,
        "sheet_name": "Sheet1",

        "output_dir": str(BASE_WORKDIR / "outputs"),
        "intermediate_dir": str(BASE_WORKDIR / "outputs" / "intermediate"),
        "figures_dir": str(BASE_WORKDIR / "outputs" / "figures"),
        "tables_dir": str(BASE_WORKDIR / "outputs" / "tables"),
        "models_dir": str(BASE_WORKDIR / "outputs" / "models"),
        "reports_dir": str(BASE_WORKDIR / "reports"),
        "qc_dir": str(BASE_WORKDIR / "outputs" / "qc"),
        "supplementary_dir": str(BASE_WORKDIR / "outputs" / "supplementary"),
    },

    "analysis": {
        "primary_binary_model": "firth",
        "secondary_binary_model": "firth",
        "legacy_penalized_logistic_for_sensitivity_only": True,
        "report_event_counts": True,
        "generate_flowchart": True,
        "rare_event_warning_threshold": 10,
        "bootstrap_random_state": 20260222
    },

    "endpoints": {
        "primary": {
            "name": "TG_HDL_high",
            "definition": "TG/HDL > 3.5",
            "tg_hdl_cutoff": 3.5,
            "units": "ratio"
        },
        "secondary": {
            "non_hdl_high_mgdl": 130,
            "ogtt120_igt_mgdl": 140
        }
    },

    "thyroid": {
        "tsh_euthyroid_low": 0.4,
        "tsh_euthyroid_high": 4.0,
        "tsh_uln": 4.0,
        "ft4_ref_low_optional": None,
        "ft4_ref_high_optional": None,
        "euthyroid_flags": {
            "tsh_only": "TSH between [tsh_euthyroid_low, tsh_euthyroid_high]",
            "strict_optional": "TSH in range AND FT4 in range"
        }
    },

    "tai_definition": {
        "A": {
            "rule": "anti_tpo > TPO_ULN",
            "tpo_uln": 34.0,
            "notes": "Primary: anti-TPO above lab ULN"
        },
        "B": {
            "rule": "anti_tpo > TPO_ULN AND tsh > TSH_ULN",
            "tpo_uln": 34.0,
            "tsh_uln": 4.0,
            "notes": "Specific: anti-TPO positive + elevated TSH"
        },
        "C": {
            "rule": "anti_tpo > TPO_HIGH_TITER",
            "high_titer": 100.0,
            "notes": "High-titer anti-TPO definition"
        },
        "D_optional": {
            "rule": "(anti_tpo > TPO_ULN) OR (anti_tg > TG_ULN)",
            "tpo_uln": 34.0,
            "tg_uln": 115.0,
            "enabled": False,
            "notes": "Sensitivity: anti-TPO OR anti-TG positivity"
        },
        "E_optional": {
            "rule": "(anti_tpo > TPO_ULN) AND (anti_tg > TG_ULN)",
            "tpo_uln": 34.0,
            "tg_uln": 115.0,
            "enabled": False,
            "notes": "Sensitivity: anti-TPO AND anti-TG positivity"
        }
    },

    "units": {
        "lipids": "mg/dL",
        "glucose": "mg/dL",
        "tsh": "mIU/L",
        "ft4": "lab_specific",
        "anti_tpo": "lab_specific",
        "anti_tg": "lab_specific"
    },

    "eligibility": {
        "required_for_primary": ["age", "anti_tpo", "tg", "hdl"],
        "required_for_secondary_non_hdl": ["age", "anti_tpo", "tc", "hdl"],
        "required_for_secondary_ogtt120": ["age", "anti_tpo", "glu120"]
    },

    "columns": {
        "id_candidates": [
            "Nr KG", "NR KG", "nr kg", "ID", "Id", "id", "patient_id", "subject_id", "Numer", "Nr"
        ],
        "age_candidates": ["Wiek", "wiek", "Age", "age"],

        "core_features": {
            "thyroid": {
                "anti_tpo": ["Anty-TPO (ATA_TPO)", "ATA_TPO", "ANTI_TPO", "ATPO", "TPO"],
                "anti_tg_optional": ["Anty-TG (p/c przeciw tyreoglobulinie) (ATG)", "ATG", "ANTI_TG"],
                "tsh": ["TSH (TSH)", "TSH", "L_TSH"],
                "ft4_optional": ["FT4 (FT4)", "FT4", "fT4", "L_FT4"],
                "ft3_optional": ["FT3 (FT3)", "FT3", "fT3", "L_FT3"],
            },
            "lipids": {
                "tg": ["Triglicerydy (TG)", "TG", "TRIGLICERYDY", "TRIG"],
                "hdl": ["HDL cholesterol (HDL)", "HDL", "L_HDL"],
                "tc_optional": ["Cholesterol całkowity (TCHOL)", "TCHOL", "CHOL_TC", "TC", "CHOL"],
                "ldl_optional": ["LDL cholesterol (LDL)", "LDL", "L_LDL"],
            },
            "glucose": {
                "glu0_optional": ["Krzywa cukrowa - 2 punktowa (L_GLU_0)", "GLU0", "GLU_0", "L_GLU_0", "L_GLU0"],
                "glu120_optional": ["Krzywa cukrowa - 2 punktowa (GLU120)", "GLU120", "GLU_120", "L_GLU_120", "L_GLU120"],
            },
            "androgens_optional": {
                "tt_optional": ["Testosteron (L_TESTOS)", "L_TESTOS", "TESTOS", "TESTOSTERON", "TESTO"],
                "ft_optional": ["Testosteron wolny (O41) (TEST-F)", "TEST-F", "TEST_F", "TESTF", "Wolny testosteron"],
                "shbg_optional": ["SHBG (SHGB)", "SHBG", "SHGB", "L_SHBG", "L_SHGB"],
                "dheas_optional": ["DHEAS (DHEA)", "DHEAS", "DHEA-S", "DHEA_S", "L_DHEA", "L_DHEAS"],
                "andro_optional": ["Androstendion (ANDRO)", "ANDRO"],
                "amh_optional": ["AMH-anty Mullerian Hormon (AMH)", "AMH", "L_AMH"],
                "lh_optional": ["LH  (LH)", "LH", "L_LH"],
                "fsh_optional": ["FSH (FSH)", "FSH", "L_FSH"],
            },
            "insulin_optional": {
                "ins0_optional": ["INS0", "INS_0", "L_INS_0", "L_INS0", "Insulina 0'", "Insulin 0"],
                "ins120_optional": ["INS120", "INS_120", "L_INS_120", "L_INS120", "Insulina 120'", "Insulin 120"],
                "insulin_optional": ["Insulina (INSUL)", "INSUL", "INSULIN", "Insulina"]
            },
            "medications_optional": {
                "lt4_use_optional": [
                    "LT4", "Levothyroxine", "L-thyroxine", "Euthyrox", "Letrox",
                    "lewotyroksyna", "leczenie_tarczycy", "thyroid_treatment"
                ]
            }
        }
    },

    "derived": {
        "tg_hdl_ratio": {"numerator": "tg", "denominator": "hdl"},
        "non_hdl": {"formula": "tc - hdl"}
    },

    "data_parsing": {
        "decimal_comma_handling": "replace_comma_with_dot",
        "range_notation_handling": {
            "pattern": "a|b",
            "default_policy": "mean",
            "alternatives_for_sensitivity": ["median", "first", "second"],
            "note": "Range-like strings are reduced to a single value for analysis."
        },
        "censoring_qualifiers_handling": {
            "patterns": ["<x", ">x"],
            "default_policy": "use_numeric_boundary",
            "note": "Inequality qualifiers mapped to the numeric boundary (e.g., '<0.01' -> 0.01).",
            "optional_sensitivity_for_antibodies": {
                "enabled": False,
                "policy": "half_lod",
                "apply_to": ["anti_tpo", "anti_tg"]
            }
        },
        "missing_tokens": ["", "nan", "none", "null", "brak", "nd", "n/d"],
        "preserve_raw_columns": True,
        "write_parse_audit": True
    },

    "transforms": {
        "winsor_q_low": 0.01,
        "winsor_q_high": 0.99,
        "log1p_candidates": ["anti_tpo", "tg", "ins0", "ins120"]
    },

    "logging": {
        "enabled": True,
        "log_file": str(BASE_WORKDIR / "outputs" / "run.log"),
        "write_qc_reports": True
    }
}

## Deep merge config and validation

In [ ]:
def deep_update(base: dict, updates: dict) -> dict:
    result = deepcopy(base)
    for k, v in updates.items():
        if isinstance(v, dict) and isinstance(result.get(k), dict):
            result[k] = deep_update(result[k], v)
        else:
            result[k] = v
    return result


def validate_config(cfg: dict) -> None:
    required_top = [
        "project", "paths", "analysis", "endpoints", "thyroid",
        "tai_definition", "columns", "derived", "data_parsing",
        "transforms", "logging"
    ]
    missing_top = [k for k in required_top if k not in cfg]
    if missing_top:
        raise ValueError(f"Missing top-level config keys: {missing_top}")

    required_path_keys = [
        "output_dir", "intermediate_dir", "figures_dir", "tables_dir",
        "models_dir", "reports_dir", "qc_dir", "supplementary_dir"
    ]
    missing_path_keys = [k for k in required_path_keys if k not in cfg["paths"]]
    if missing_path_keys:
        raise ValueError(f"Missing cfg['paths'] keys: {missing_path_keys}")

    if cfg["analysis"]["primary_binary_model"] not in {"firth", "logistic", "exact"}:
        raise ValueError("Unsupported primary_binary_model in config.")

    if cfg["analysis"]["secondary_binary_model"] not in {"firth", "logistic", "exact"}:
        raise ValueError("Unsupported secondary_binary_model in config.")

## Load/create config and resolve input file

In [ ]:
def load_or_create_config(config_path: Path, defaults: dict) -> dict:

    if config_path.exists():
        try:
            raw_text = config_path.read_text(encoding="utf-8").strip()

            if raw_text == "":
                raise json.JSONDecodeError("Empty config file", raw_text, 0)

            existing = json.loads(raw_text)
            cfg = deep_update(defaults, existing)
            print(f"Loaded config: {config_path}")

        except (json.JSONDecodeError, OSError):
            print(f"Warning: Existing config file {config_path} is malformed or empty. Recreating with defaults.")
            config_path.parent.mkdir(parents=True, exist_ok=True)
            cfg = deepcopy(defaults)
            with open(config_path, "w", encoding="utf-8") as f:
                json.dump(cfg, f, ensure_ascii=False, indent=2)
            print(f"Created config: {config_path}")

    else:
        config_path.parent.mkdir(parents=True, exist_ok=True)
        cfg = deepcopy(defaults)
        with open(config_path, "w", encoding="utf-8") as f:
            json.dump(cfg, f, ensure_ascii=False, indent=2)
        print(f"Created config: {config_path}")

    cfg["paths"]["input_xlsx"] = resolve_uploaded_xlsx()
    validate_config(cfg)


    with open(config_path, "w", encoding="utf-8") as f:
        json.dump(cfg, f, ensure_ascii=False, indent=2)

    return cfg


CFG = load_or_create_config(CONFIG_PATH, CFG_DEFAULT)

Created config: /content/config.json


## Creating catalogs

In [ ]:
def ensure_dirs(cfg: dict) -> None:
    dir_keys = [
        "output_dir", "intermediate_dir", "figures_dir", "tables_dir",
        "models_dir", "reports_dir", "qc_dir", "supplementary_dir"
    ]
    for key in dir_keys:
        Path(cfg["paths"][key]).mkdir(parents=True, exist_ok=True)


ensure_dirs(CFG)

## Logging

In [ ]:
def setup_logging(cfg: dict) -> None:
    if not cfg.get("logging", {}).get("enabled", False):
        return

    log_file = Path(cfg["logging"]["log_file"])
    log_file.parent.mkdir(parents=True, exist_ok=True)

    root_logger = logging.getLogger()
    root_logger.setLevel(logging.INFO)

    if root_logger.handlers:
        has_file_handler = any(
            isinstance(h, logging.FileHandler) and Path(getattr(h, "baseFilename", "")) == log_file
            for h in root_logger.handlers
        )
        if not has_file_handler:
            file_handler = logging.FileHandler(log_file, encoding="utf-8")
            file_handler.setFormatter(logging.Formatter("%(asctime)s | %(levelname)s | %(message)s"))
            root_logger.addHandler(file_handler)
        return

    logging.basicConfig(
        level=logging.INFO,
        format="%(asctime)s | %(levelname)s | %(message)s",
        handlers=[
            logging.FileHandler(log_file, encoding="utf-8"),
            logging.StreamHandler()
        ],
    )


setup_logging(CFG)

logging.info("Logging initialized.")
logging.info("Project: %s v%s", CFG["project"]["name"], CFG["project"]["version"])
logging.info("Notebook: %s", CFG["project"]["notebook"])
logging.info("Python: %s", platform.python_version())
logging.info("pandas: %s", pd.__version__)
logging.info("numpy: %s", np.__version__)
logging.info("Config path: %s", str(CONFIG_PATH))
logging.info("Resolved input xlsx: %s", CFG["paths"]["input_xlsx"])

INFO:root:Logging initialized.
INFO:root:Project: PCOS_TAI_cross_sectional v0.2.1
INFO:root:Notebook: S00_config
INFO:root:Python: 3.12.12
INFO:root:pandas: 2.2.2
INFO:root:numpy: 2.0.2
INFO:root:Config path: /content/config.json
INFO:root:Resolved input xlsx: /content/patients_results.xlsx


## Seed

In [ ]:
RANDOM_SEED = int(CFG["project"]["random_seed"])

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

logging.info("Random seed set to %s", RANDOM_SEED)

INFO:root:Random seed set to 20260222


## Regex and column name normalization

In [ ]:
_NUM_RE = re.compile(r"[-+]?\d*\.?\d+(?:[eE][-+]?\d+)?")

def normalize_colname(s: str) -> str:
    return re.sub(r"\s+", " ", str(s).strip().lower())

## Cell parser with QC flag

In [ ]:
def parse_numeric_cell_with_flag(
    x: object,
    range_policy: str = "mean",
    missing_tokens: Optional[Sequence[str]] = None
) -> Tuple[float, str]:
    """
    Parse heterogeneous numeric laboratory cells into a float and a parse flag.

    Flags:
    - numeric
    - missing
    - missing_token
    - range
    - left_censored
    - right_censored
    - parsed
    - unparsable
    """
    if missing_tokens is None:
        missing_tokens = CFG.get("data_parsing", {}).get("missing_tokens", [])

    missing_tokens_lower = {str(t).lower() for t in missing_tokens}

    if x is None:
        return float("nan"), "missing"

    if isinstance(x, (int, float, np.integer, np.floating)):
        try:
            if np.isnan(x):
                return float("nan"), "missing"
        except Exception:
            pass
        return float(x), "numeric"

    s = str(x).strip()
    if s == "" or s.lower() in missing_tokens_lower:
        return float("nan"), "missing_token"

    s = s.replace(" ", "")
    s = s.replace(";", "|")

    if "|" in s:
        parts = [p for p in s.split("|") if p]
        vals = []
        for p in parts:
            p2 = p.replace(",", ".")
            if p2.startswith("<") or p2.startswith(">"):
                p2 = p2[1:]
            m = _NUM_RE.search(p2)
            if m:
                vals.append(float(m.group(0)))

        if len(vals) == 0:
            return float("nan"), "unparsable"
        if len(vals) == 1:
            return float(vals[0]), "range"

        if range_policy == "median":
            return float(np.median(vals)), "range"
        if range_policy == "first":
            return float(vals[0]), "range"
        if range_policy == "second":
            return float(vals[1]), "range"

        return float(np.mean(vals)), "range"

    s2 = s.replace(",", ".")

    if s2.startswith("<"):
        s2_num = s2[1:]
        m = _NUM_RE.search(s2_num)
        if not m:
            return float("nan"), "unparsable"
        return float(m.group(0)), "left_censored"

    if s2.startswith(">"):
        s2_num = s2[1:]
        m = _NUM_RE.search(s2_num)
        if not m:
            return float("nan"), "unparsable"
        return float(m.group(0)), "right_censored"

    m = _NUM_RE.search(s2)
    if not m:
        return float("nan"), "unparsable"

    try:
        return float(m.group(0)), "parsed"
    except Exception:
        return float("nan"), "unparsable"

## Backward compatible parser

In [ ]:
def parse_numeric_cell(
    x: object,
    range_policy: str = "mean",
    missing_tokens: Optional[Sequence[str]] = None
) -> float:
    val, _flag = parse_numeric_cell_with_flag(
        x=x,
        range_policy=range_policy,
        missing_tokens=missing_tokens
    )
    return val

## Batch conversion and parsing audit

In [ ]:
def coerce_numeric_series(
    ser: pd.Series,
    range_policy: str = "mean"
) -> pd.Series:
    return ser.apply(
        lambda v: parse_numeric_cell(v, range_policy=range_policy)
    ).astype("float64")


def coerce_numeric_series_with_flags(
    ser: pd.Series,
    range_policy: str = "mean"
) -> pd.DataFrame:
    parsed = ser.apply(lambda v: parse_numeric_cell_with_flag(v, range_policy=range_policy))
    out = pd.DataFrame({
        "value": parsed.apply(lambda x: x[0]).astype("float64"),
        "parse_flag": parsed.apply(lambda x: x[1]).astype("string")
    })
    return out


def parsing_audit_report(
    ser: pd.Series,
    range_policy: str = "mean"
) -> pd.DataFrame:
    audit_df = coerce_numeric_series_with_flags(ser, range_policy=range_policy)
    rep = (
        audit_df["parse_flag"]
        .value_counts(dropna=False)
        .rename_axis("parse_flag")
        .reset_index(name="n")
    )
    rep["pct"] = (rep["n"] / len(audit_df) * 100).round(2)
    return rep

## Analytical helpers

In [ ]:
def find_column_by_candidates(df: pd.DataFrame, candidates: Sequence[str]) -> Optional[str]:
    cols = list(df.columns)
    norm_map = {normalize_colname(c): c for c in cols}

    for cand in candidates:
        if cand in cols:
            return cand
        nc = normalize_colname(cand)
        if nc in norm_map:
            return norm_map[nc]
    return None


def safe_divide(a: pd.Series, b: pd.Series) -> pd.Series:
    a = pd.to_numeric(a, errors="coerce")
    b = pd.to_numeric(b, errors="coerce").replace({0: np.nan})
    return a / b


def safe_log1p(x: pd.Series) -> pd.Series:
    x = pd.to_numeric(x, errors="coerce")
    x = x.where(x >= 0, np.nan)
    return np.log1p(x)

## Winsorization

In [ ]:
def winsorize_series(
    x: pd.Series,
    q_low: float,
    q_high: float
) -> pd.Series:
    x = pd.to_numeric(x, errors="coerce")
    lo = x.quantile(q_low)
    hi = x.quantile(q_high)
    return x.clip(lower=lo, upper=hi)


def winsorize_series_with_info(
    x: pd.Series,
    q_low: float,
    q_high: float
) -> Tuple[pd.Series, Dict[str, Any]]:
    x = pd.to_numeric(x, errors="coerce")
    lo = x.quantile(q_low)
    hi = x.quantile(q_high)
    xw = x.clip(lower=lo, upper=hi)

    info = {
        "q_low": q_low,
        "q_high": q_high,
        "lower_bound": float(lo) if pd.notna(lo) else np.nan,
        "upper_bound": float(hi) if pd.notna(hi) else np.nan,
        "n_clipped_low": int((x < lo).sum()) if pd.notna(lo) else 0,
        "n_clipped_high": int((x > hi).sum()) if pd.notna(hi) else 0,
        "n_nonmissing": int(x.notna().sum())
    }
    return xw, info

## Missingness report

In [ ]:
def missingness_report(df: pd.DataFrame) -> pd.DataFrame:
    rep = pd.DataFrame({
        "n_missing": df.isna().sum(),
        "n_nonmissing": df.notna().sum(),
        "pct_missing": (df.isna().mean() * 100).round(2),
        "dtype": df.dtypes.astype(str),
    }).sort_values(["pct_missing", "n_missing"], ascending=False)
    return rep

## Config and environment snapshot

In [ ]:
def save_config_snapshot(cfg: dict) -> None:
    out = Path(cfg["paths"]["reports_dir"]) / "config_snapshot.json"
    snapshot = deepcopy(cfg)
    snapshot["_snapshot_metadata"] = {
        "saved_at": datetime.utcnow().isoformat() + "Z",
        "python_version": platform.python_version(),
        "pandas_version": pd.__version__,
        "numpy_version": np.__version__,
    }

    with open(out, "w", encoding="utf-8") as f:
        json.dump(snapshot, f, ensure_ascii=False, indent=2)

    logging.info("Saved config snapshot: %s", str(out))


def save_environment_report(cfg: dict) -> None:
    out = Path(cfg["paths"]["reports_dir"]) / "environment_report.json"
    payload = {
        "saved_at": datetime.utcnow().isoformat() + "Z",
        "python_version": platform.python_version(),
        "platform": platform.platform(),
        "pandas_version": pd.__version__,
        "numpy_version": np.__version__,
        "project_name": cfg["project"]["name"],
        "project_version": cfg["project"]["version"],
        "random_seed": cfg["project"]["random_seed"],
        "input_xlsx": cfg["paths"]["input_xlsx"]
    }
    with open(out, "w", encoding="utf-8") as f:
        json.dump(payload, f, ensure_ascii=False, indent=2)
    logging.info("Saved environment report: %s", str(out))

## Recording reports

In [ ]:
save_config_snapshot(CFG)
save_environment_report(CFG)

/tmp/ipykernel_16152/3319211097.py:5: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "saved_at": datetime.utcnow().isoformat() + "Z",
INFO:root:Saved config snapshot: /content/reports/config_snapshot.json
/tmp/ipykernel_16152/3319211097.py:20: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "saved_at": datetime.utcnow().isoformat() + "Z",
INFO:root:Saved environment report: /content/reports/environment_report.json


## Input file validation

In [ ]:
input_path = Path(CFG["paths"]["input_xlsx"])

if not input_path.exists():
    raise FileNotFoundError(f"Resolved input_xlsx does not exist: {input_path}")

logging.info("Input file exists: %s", input_path.exists())
logging.info("Input file size (bytes): %s", input_path.stat().st_size)

INFO:root:Input file exists: True
INFO:root:Input file size (bytes): 467021


## Summary

In [ ]:
print("CFG initialized.")
print("Project:", CFG["project"]["name"], CFG["project"]["version"])
print("Notebook:", CFG["project"]["notebook"])
print("Random seed:", RANDOM_SEED)
print("Config path:", str(CONFIG_PATH))
print("Using input_xlsx:", CFG["paths"]["input_xlsx"])
print("Input exists:", input_path.exists())
print("Reports dir:", CFG["paths"]["reports_dir"])
print("QC dir:", CFG["paths"]["qc_dir"])

CFG initialized.
Project: PCOS_TAI_cross_sectional 0.2.1
Notebook: S00_config
Random seed: 20260222
Config path: /content/config.json
Using input_xlsx: /content/patients_results.xlsx
Input exists: True
Reports dir: /content/reports
QC dir: /content/outputs/qc
